In [1]:
# configuration, données, modèle
import glob, os, time, math, random, json
import numpy as np, pandas as pd, torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "/kaggle/working/results"; os.makedirs(OUT_DIR, exist_ok=True)
print("device :", device)

df = pd.read_csv(glob.glob("/kaggle/input/**/dataset.csv", recursive=True)[0])
corpus = df[df["score"] >= 6.5]["text"].tolist()          # articles très pertinents (classe « élevé »)
random.shuffle(corpus)
n_val = max(10, len(corpus) // 10)
val_texts, train_texts = corpus[:n_val], corpus[n_val:]
print(len(df), "articles |", len(train_texts), "train |", len(val_texts), "validation")

MODEL = "aubmindlab/aragpt2-base"
tok = GPT2TokenizerFast.from_pretrained(MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = GPT2LMHeadModel.from_pretrained(MODEL).to(device)
print("paramètres :", sum(p.numel() for p in model.parameters()))

device : cuda
570 articles | 104 train | 11 validation


vocab.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  553MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


paramètres : 134994432


In [2]:
# blocs de texte et perplexité avant fine-tuning
BLOCK = 256
def make_blocks(texts):
    ids = []
    for t in texts: ids += tok(t)["input_ids"] + [tok.eos_token_id]
    n = (len(ids) // BLOCK) * BLOCK
    return torch.tensor(ids[:n]).view(-1, BLOCK)

train_blocks, val_blocks = make_blocks(train_texts), make_blocks(val_texts)
print("blocs train / validation :", tuple(train_blocks.shape), tuple(val_blocks.shape))

@torch.no_grad()
def perplexity(m, blocks, bs=8):
    m.eval(); tot, n = 0.0, 0
    for i in range(0, len(blocks), bs):
        b = blocks[i:i+bs].to(device)
        tot += m(b, labels=b).loss.item() * b.numel(); n += b.numel()
    return math.exp(tot / n)

ppl_before = perplexity(model, val_blocks)
print(f"perplexité avant fine-tuning : {ppl_before:.1f}")

blocs train / validation : (236, 256) (30, 256)


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


perplexité avant fine-tuning : 228.3


In [3]:
#  fine-tuning
EPOCHS, BS, LR = 3, 4, 5e-5
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * math.ceil(len(train_blocks) / BS)
sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
scaler = torch.amp.GradScaler("cuda")
hist = []
torch.cuda.synchronize(); t0 = time.time()
for ep in range(EPOCHS):
    model.train(); order = torch.randperm(len(train_blocks)); tot = 0.0
    for i in range(0, len(order), BS):
        b = train_blocks[order[i:i+BS]].to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(b, labels=b).loss
        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * len(b)
    ppl = perplexity(model, val_blocks)
    hist.append((tot / len(train_blocks), ppl))
    print(f"epoch {ep+1}/{EPOCHS} | loss train {hist[-1][0]:.3f} | perplexité val {ppl:.1f} | {time.time()-t0:.0f} s")
torch.cuda.synchronize(); train_time = time.time() - t0
model.save_pretrained(f"{OUT_DIR}/aragpt2_finetuned"); tok.save_pretrained(f"{OUT_DIR}/aragpt2_finetuned")

/tmp/ipykernel_49/2488500977.py:17: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


epoch 1/3 | loss train 7.183 | perplexité val 107.4 | 10 s
epoch 2/3 | loss train 5.388 | perplexité val 79.6 | 20 s
epoch 3/3 | loss train 4.965 | perplexité val 72.8 | 29 s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/results/aragpt2_finetuned/tokenizer_config.json',
 '/kaggle/working/results/aragpt2_finetuned/tokenizer.json')

In [4]:
# génération, BLEU, sauvegarde
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
base = GPT2LMHeadModel.from_pretrained(MODEL).to(device).eval()      # modèle avant fine-tuning
model.eval()

def generate(m, prompt, max_new=120, sample=True):
    x = tok(prompt, return_tensors="pt").to(device)
    kw = dict(do_sample=True, top_p=0.92, top_k=50, temperature=0.8) if sample else dict(do_sample=False)
    out = m.generate(**x, max_new_tokens=max_new, repetition_penalty=1.2, no_repeat_ngram_size=3,
                     pad_token_id=tok.eos_token_id, **kw)
    return tok.decode(out[0], skip_special_tokens=True)

PROMPTS = ["يشهد قطاع الذكاء الاصطناعي تطورا متسارعا",
           "تعمل الشركات التقنية على تطوير نماذج لغوية",
           "يحذر الخبراء من مخاطر الذكاء الاصطناعي على سوق العمل"]
samples = []
torch.manual_seed(SEED)
for p in PROMPTS:
    a, b = generate(base, p), generate(model, p)
    samples.append({"prompt": p, "avant": a, "apres": b})
    print("=" * 70); print("PROMPT :", p); print("\n[avant]", a); print("\n[après]", b)

sm = SmoothingFunction().method1
def bleu_eval(m, texts, prompt_tok=30, ref_tok=60):
    b1, b2 = [], []
    for t in texts:
        ids = tok(t)["input_ids"]
        if len(ids) < prompt_tok + ref_tok: continue
        prompt = tok.decode(ids[:prompt_tok]); ref = tok.decode(ids[prompt_tok:prompt_tok + ref_tok]).split()
        x = tok(prompt, return_tensors="pt").to(device)
        out = m.generate(**x, max_new_tokens=ref_tok, do_sample=False, repetition_penalty=1.2,
                         no_repeat_ngram_size=3, pad_token_id=tok.eos_token_id)
        hyp = tok.decode(out[0][x["input_ids"].shape[1]:], skip_special_tokens=True).split()
        b1.append(sentence_bleu([ref], hyp, weights=(1, 0, 0, 0), smoothing_function=sm))
        b2.append(sentence_bleu([ref], hyp, weights=(.5, .5, 0, 0), smoothing_function=sm))
    return float(np.mean(b1)), float(np.mean(b2)), len(b1)

bleu_before, bleu_after = bleu_eval(base, val_texts), bleu_eval(model, val_texts)
ppl_after = hist[-1][1]
print(f"\nperplexité : {ppl_before:.1f} -> {ppl_after:.1f}")
print("BLEU-1 / BLEU-2 / n avant :", [round(v, 3) for v in bleu_before])
print("BLEU-1 / BLEU-2 / n après :", [round(v, 3) for v in bleu_after])

json.dump({"model": MODEL, "train_articles": len(train_texts), "val_articles": len(val_texts),
           "blocks_train": len(train_blocks), "epochs": EPOCHS, "lr": LR, "batch": BS, "train_time_s": train_time,
           "ppl_before": ppl_before, "ppl_after": ppl_after, "bleu_before": bleu_before, "bleu_after": bleu_after,
           "history": hist, "samples": samples}, open(f"{OUT_DIR}/results_gpt2.json", "w"), ensure_ascii=False, indent=1)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PROMPT : يشهد قطاع الذكاء الاصطناعي تطورا متسارعا

[avant] يشهد قطاع الذكاء الاصطناعي تطورا متسارعا ، حيث باتت الشركات تركز على تقديم ابتكارات جديدة في مجال الأجهزة الذكية بهدف الاستفادة من النمو المستمر للتكنولوجيا المستخدمة في هذا المجال .وذكرت مجلة " بي سي وورلد سيرفيسز " إن هذه الخطوة ستساعد على تعزيز مكانة الشركة التي تأسست عام 2005 لتصبح واحدة من أكبر شركات البرمجيات حول العالم .وستكون هواوي رائدة صناعة الهواتف الذكية على مستوى العالم ، إلى جانب شركات أخرى مثل شركة " آبل " و " سامسونغ " و ال جي " وغيرها .

[après] يشهد قطاع الذكاء الاصطناعي تطورا متسارعا في عدد الأجهزة العاملة بنظام التشغيل، حيث شهد نموا كبيرا خلال السنوات الماضية بنسبة 15% مقارنة بالناتجانية التي شهدها العام 2026.
وأشارت الشركة أن التحول الرقمي ساهم في تحسين حياة الناس ، وتطوير حياتهم بشكل كبيرًومن جانبه قال المدير التنفيذي ل « وكانت شركة "س آند بي سي إن آي" الأميركية العملاقة المتخصصة في مجال تقنيات الذكاء الاصطناعي، ستيف غولدمان ساكس أن التكنولوجيا باتت واقعا ملموسا بعد إطلاق أول نظام ذكاء اصطناعي في الأسواق ع